# Samping for final report

# 1. Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import joblib
import time

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# import gradient boosting model
from sklearn.ensemble import GradientBoostingClassifier

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Global Configuration
RANDOM_STATE = 22122025
np.random.seed(RANDOM_STATE)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"Random seed set to: {RANDOM_STATE}")

✓ Random seed set to: 22122025


In [2]:
number_of_sample = 10

In [3]:
MODE = 'colab'  # 'local' or 'colab'
# MODE = 'local'

if MODE == 'colab':
    from google.colab import drive

    # Mount Google Drive
    drive.mount('/content/drive')

    head_path = '/content/drive/MyDrive/thesis/'


    final_features_path = head_path + r"/data/final_for_modeling/temp/final_features.pkl"
    final_bin_path = head_path + r"/data/final_for_modeling/temp/final_bin.pkl"
    embedding_time_report_path = head_path + r"/data/embedding_time_report.csv"
    model_training_report_path = head_path + r"/data/model_training_report.csv"

    benchmark_model_path = head_path + r"/data/models/baseline_models/benchmark_model.pkl"

    train_df_path = head_path + r"/data/final_for_modeling/train_ad_break_level_data.parquet"
    test_df_path = head_path + r"/data/final_for_modeling/test_ad_break_level_data.parquet"
    segment_df_path = head_path + r"/data/final_for_modeling/segment_df.parquet"
    ts2vec_df_path = head_path + r"/data/final_for_modeling/bin_ts2vec_emb_df.parquet"
    femba_df_path = head_path + r"/data/final_for_modeling/bin_femba_emb_df.parquet"

else:
    final_features_path = r'../../data/final_for_modeling/temp/final_features.pkl'
    final_bin_path = r'../../data/final_for_modeling/temp/final_bin.pkl'
    embedding_time_report_path = r'../../data/embedding_time_report.csv'
    model_training_report_path = r'../../data/model_training_report.csv'

    benchmark_model_path = r"../../models/baseline_models/benchmark_model.pkl"

    train_df_path = r'../../data/final_for_modeling/train_ad_break_level_data.parquet'
    test_df_path = r'../../data/final_for_modeling/test_ad_break_level_data.parquet'
    segment_df_path = r'../../data/final_for_modeling/segment_df.parquet'
    ts2vec_df_path = r'../../data/final_for_modeling/bin_ts2vec_emb_df.parquet'
    femba_df_path = r'../../data/final_for_modeling/bin_femba_emb_df.parquet'

Mounted at /content/drive


# 2. Load data

In [4]:
final_features = joblib.load(final_features_path)
final_bin = joblib.load(final_bin_path)

# 3. Run Embedding Training

In [5]:
# Define feature columns (excluding ID columns)
feature_cols = [
    c for c in final_bin.columns
    if c not in ["ad_break_session_key", "bin_order"]
]
print(f"Number of features: {len(feature_cols)}")

#num_series × time_steps × num_features
sequences = []
keys = []

for key, g in final_bin.groupby("ad_break_session_key"):
    g = g.sort_values("bin_order")
    X = g[feature_cols].values  # shape: (T, F)
    sequences.append(X)
    keys.append(key)

len(sequences)

Number of features: 49


2835

## 3.1 TS2Vec Embedding Training

In [6]:
%pip install ts2vec

from ts2vec import TS2Vec
import torch

In [7]:
 # Pad sequences to same length and convert to 3D array
max_len = max(seq.shape[0] for seq in sequences)
padded_sequences = np.array([
    np.pad(seq, ((0, max_len - seq.shape[0]), (0, 0)), mode='constant')
    for seq in sequences
])

ts2vec_report = {}

for sample in range(number_of_sample):
    print(f"sample number {sample+1} training...")

    #==================== Timing the training process ====================
    start = time.perf_counter()
    model = TS2Vec(
        input_dims=padded_sequences.shape[2],  # 49 for now
        output_dims=128,
        hidden_dims=64,
        depth=10,
        device= 'cuda' if torch.cuda.is_available() else 'cpu',
    )

    model.fit(
        padded_sequences,  # a 3D numpy array: (num_series, time_steps, features)
        n_epochs=30
    )
    #=====================================================================
    fit_time = time.perf_counter() - start


    # ===================== Timing the embedding process ====================
    start = time.perf_counter()
    embeddings = model.encode(padded_sequences)
    # =====================================================================
    encode_time = time.perf_counter() - start

    ################################################################################
    ts2vec_report[sample+1] = {
        'fit_time': fit_time,
        'encode_time': encode_time,
        'total_time_s': fit_time + encode_time
    }

    del(model)
    del(embeddings)

sample number 1 training...
sample number 2 training...
sample number 3 training...
sample number 4 training...
sample number 5 training...
sample number 6 training...
sample number 7 training...
sample number 8 training...
sample number 9 training...
sample number 10 training...


In [8]:
ts2vec_report_df = pd.DataFrame.from_dict(ts2vec_report, orient='index').rename_axis('sample').reset_index()
# time report
print("=" * 40)
print("TS2Vec Feature Extraction Time Report")
print(ts2vec_report_df.to_string(index=False, justify='center', float_format="{:.3f}".format))

TS2Vec Feature Extraction Time Report
 sample  fit_time  encode_time  total_time_s
    1    393.217     1.826        395.043   
    2    367.386     1.470        368.857   
    3    365.669     1.507        367.176   
    4    359.444     2.288        361.732   
    5    359.005     1.685        360.690   
    6    356.759     1.453        358.212   
    7    373.370     2.511        375.882   
    8    371.370     1.538        372.908   
    9    365.011     1.530        366.541   
   10    360.834     1.474        362.308   


## 3.2 Femba Embedding Training

In [9]:
%pip install torch

import torch
import torch.nn as nn

In [10]:
# ================= Padding (same as TS2Vec) =================
max_len = max(seq.shape[0] for seq in sequences)
padded_sequences = np.array([
    np.pad(seq, ((0, max_len - seq.shape[0]), (0, 0)), mode='constant')
    for seq in sequences
])

X = torch.tensor(padded_sequences, dtype=torch.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
X = X.to(device)

# ================= FEMBA Model =================
class FEMBA(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, emb_dim=128):
        super().__init__()
        self.encoder_rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.embedding = nn.Linear(hidden_dim, emb_dim)

        self.decoder_rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, h = self.encoder_rnn(x)
        z = self.embedding(h[-1])

        z_rep = z.unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _ = self.decoder_rnn(z_rep)
        x_hat = self.output_layer(dec_out)
        return x_hat, z


femba_report = {}

for sample in range(number_of_sample):
    print(f"sample number {sample+1} training...")

    # ================= Model, Optimizer, Loss =================
    model = FEMBA(
        input_dim=X.shape[2],
        hidden_dim=64,
        emb_dim=128
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    # ================= Training =================
    start = time.perf_counter()

    epochs = 30
    batch_size = 32

    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(X.size(0))
        for i in range(0, X.size(0), batch_size):
            idx = perm[i:i+batch_size]
            batch = X[idx]

            optimizer.zero_grad()
            x_hat, _ = model(batch)
            loss = criterion(x_hat, batch)
            loss.backward()
            optimizer.step()

    fit_time = time.perf_counter() - start

    # ================= Embedding =================
    start = time.perf_counter()
    model.eval()
    with torch.no_grad():
        _, embeddings = model(X)

    embeddings = embeddings.cpu().numpy()  # (num_series, emb_dim)

    encode_time = time.perf_counter() - start

    ################################################################################
    femba_report[sample+1] = {
        'fit_time': fit_time,
        'encode_time': encode_time,
        'total_time_s': fit_time + encode_time
    }

    # del(model)
    # del(embeddings)


sample number 1 training...
sample number 2 training...
sample number 3 training...
sample number 4 training...
sample number 5 training...
sample number 6 training...
sample number 7 training...
sample number 8 training...
sample number 9 training...
sample number 10 training...


In [11]:
femba_report

{1: {'fit_time': 23.314120837000246,
  'encode_time': 0.11342954499968982,
  'total_time_s': 23.427550381999936},
 2: {'fit_time': 21.736055189000126,
  'encode_time': 0.13183917900005326,
  'total_time_s': 21.86789436800018},
 3: {'fit_time': 23.062662353000178,
  'encode_time': 0.08595451700011836,
  'total_time_s': 23.148616870000296},
 4: {'fit_time': 23.20438553399981,
  'encode_time': 0.1036018210002112,
  'total_time_s': 23.307987355000023},
 5: {'fit_time': 21.63879821599994,
  'encode_time': 0.08672949399988283,
  'total_time_s': 21.725527709999824},
 6: {'fit_time': 23.14184754000007,
  'encode_time': 0.09417813100026251,
  'total_time_s': 23.236025671000334},
 7: {'fit_time': 22.234013084000253,
  'encode_time': 0.1555942100003449,
  'total_time_s': 22.389607294000598},
 8: {'fit_time': 22.696033719999832,
  'encode_time': 0.09954968200008807,
  'total_time_s': 22.79558340199992},
 9: {'fit_time': 23.16467916600004,
  'encode_time': 0.08794348199990054,
  'total_time_s': 23.

In [12]:
femba_report_df = pd.DataFrame.from_dict(femba_report, orient='index').rename_axis('sample').reset_index()
# time report
print("=" * 40)
print("FEMBA Feature Extraction Time Report")
print(femba_report_df.to_string(index=False, justify='center', float_format="{:.3f}".format))

FEMBA Feature Extraction Time Report
 sample  fit_time  encode_time  total_time_s
    1     23.314     0.113         23.428   
    2     21.736     0.132         21.868   
    3     23.063     0.086         23.149   
    4     23.204     0.104         23.308   
    5     21.639     0.087         21.726   
    6     23.142     0.094         23.236   
    7     22.234     0.156         22.390   
    8     22.696     0.100         22.796   
    9     23.165     0.088         23.253   
   10     23.068     0.122         23.190   


# 4. Embedding Time Report

In [13]:
embedding_time_report_df = femba_report_df.merge(
                                    ts2vec_report_df,
                                    left_on='sample',
                                    right_on='sample',
                                    suffixes=('_femba', '_ts2vec')
                                ).round(3)

print(embedding_time_report_df.to_string(index=False, justify='center', float_format="{:.3f}".format))

 sample  fit_time_femba  encode_time_femba  total_time_s_femba  fit_time_ts2vec  encode_time_ts2vec  total_time_s_ts2vec
    1        23.314           0.113               23.428           393.217             1.826              395.043       
    2        21.736           0.132               21.868           367.386             1.470              368.857       
    3        23.063           0.086               23.149           365.669             1.507              367.176       
    4        23.204           0.104               23.308           359.444             2.288              361.732       
    5        21.639           0.087               21.726           359.005             1.685              360.690       
    6        23.142           0.094               23.236           356.759             1.453              358.212       
    7        22.234           0.156               22.390           373.370             2.511              375.882       
    8        22.696           0.

Exporting the final report

In [14]:
embedding_time_report_df.to_csv(
    embedding_time_report_path,
    index=False)

# 5. Model Training on Final Embeddings

### Prepare Data

In [15]:
# Load train and val data
train_df = pd.read_parquet(train_df_path)
test_df = pd.read_parquet(test_df_path)

segment_df = pd.read_parquet(segment_df_path)
ts2vec_df = pd.read_parquet(ts2vec_df_path)
femba_df = pd.read_parquet(femba_df_path)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"-----------------------------------")
print(f"Segment shape: {segment_df.shape}")
print(f"TS2Vec Emb shape: {ts2vec_df.shape}")
print(f"FEMBA Emb shape: {femba_df.shape}")
print(f"-----------------------------------")

target_col = 'unaided_brand_recall'

drop_cols = [
    'ad_break_session_key', 'participant_id_calculated',
]

dummy_columns = ['category', 'brand',  'content_watched',
                'device', 'seen_ad_before_score', 'age_group', 'gender']

def custom_get_dummies(df, columns = dummy_columns):
    """Custom one-hot encoding for categorical variables."""
    final_df = df.copy()
    final_df = pd.get_dummies(final_df, columns=columns)
    return final_df

def custom_load(df, train_df=train_df, test_df=test_df):
    """
    Custom function to load and preprocess the data for modeling.

    outputs:
        X_train_df, X_test_df, y_train, y_test
    """

    # make the big dataframe
    df_train = train_df.merge(df, on='ad_break_session_key', how='left')
    df_test = test_df.merge(df, on='ad_break_session_key', how='left')

    # TEMP Drop null rows
    df_train = df_train.dropna(axis=0)
    df_test = df_test.dropna(axis=0)

    # dummy encoding
    df_train = custom_get_dummies(df_train).drop(columns='seen_ad_before_score_Unknown')
    df_test = custom_get_dummies(df_test).drop(columns='seen_ad_before_score_Unknown')

    # drop key columns
    df_train = df_train.drop(columns=drop_cols)
    df_test = df_test.drop(columns=drop_cols)

    # split into X and y
    X_train_df = df_train.drop(columns=[target_col])
    y_train = df_train[target_col]

    X_test_df = df_test.drop(columns=[target_col])
    y_test = df_test[target_col]

    print(f" Test shape: {df_test.shape}, Train shape: {df_train.shape}")
    print(f"-----------------------------------")

    return X_train_df, X_test_df, y_train, y_test

# Load classic mean with Custom load
X_classic_mean_train, X_classic_mean_test, y_classic_mean_train, yclassic_mean_test = custom_load(segment_df)

# Ts2Vec Embeddings
X_ts2vec_train, X_ts2vec_test, y_ts2vec_train, y_ts2vec_test = custom_load(ts2vec_df)

# FEMBA Embeddings
X_femba_train, X_femba_test, y_femba_train, y_femba_test = custom_load(femba_df)


# create a dictionary to hold all datasets
datasets_info = {
    "classic_mean": {
        "X_train": X_classic_mean_train,
        "X_test": X_classic_mean_test,
        "y_train": y_classic_mean_train,
        "y_test": yclassic_mean_test
    },
    "ts2vec": {
        "X_train": X_ts2vec_train,
        "X_test": X_ts2vec_test,
        "y_train": y_ts2vec_train,
        "y_test": y_ts2vec_test
    },
    "femba": {
        "X_train": X_femba_train,
        "X_test": X_femba_test,
        "y_train": y_femba_train,
        "y_test": y_femba_test
    }
}

Train shape: (1970, 22)
Test shape: (865, 22)
-----------------------------------
Segment shape: (2835, 50)
TS2Vec Emb shape: (2835, 129)
FEMBA Emb shape: (2835, 129)
-----------------------------------
 Test shape: (865, 145), Train shape: (1970, 145)
-----------------------------------
 Test shape: (865, 224), Train shape: (1970, 224)
-----------------------------------
 Test shape: (865, 224), Train shape: (1970, 224)
-----------------------------------


### Training

In [17]:
# Storage for all results
results = {}

for sample in range(number_of_sample):
    print(f"sample number {sample+1} training...")

    for name in ["classic_mean", "ts2vec", "femba"]:
        # reset benchmark model for each run
        benchmark_model = joblib.load(benchmark_model_path)

        print(f"      - Training on {name.upper()} representation")

        # Get pre-loaded data (already preprocessed)
        X_train = datasets_info[name]["X_train"]
        X_test = datasets_info[name]["X_test"]
        y_train = datasets_info[name]["y_train"]
        y_test = datasets_info[name]["y_test"]

        # Train and time
        start_time = time.time()
        benchmark_model.fit(X_train, y_train)
        train_time = time.time() - start_time

        # Generate predictions
        y_pred_test = benchmark_model.predict(X_test)
        y_proba_test = benchmark_model.predict_proba(X_test)[:, 1]

        # Also get training predictions for overfitting check
        y_pred_train = benchmark_model.predict(X_train)
        y_proba_train = benchmark_model.predict_proba(X_train)[:, 1]

        # Store results with nested dictionary structure
        if name not in results:
            results[name] = {}

        results[name][sample + 1] = {
            'train_time': train_time
        }



print("\n" + "=" * 60)
print("ALL MODELS TRAINED SUCCESSFULLY")
print("=" * 60)

sample number 1 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 2 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 3 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 4 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 5 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 6 training...
      - Training on CLASSIC_MEAN representation
      - Training on TS2VEC representation
      - Training on FEMBA representation
sample number 7 training...
      - Training o

# 6. Final Model Training Report

In [18]:
classic_mean_df = pd.DataFrame.from_dict(results['classic_mean'], orient='index').rename_axis('sample').reset_index()
ts2vec_df = pd.DataFrame.from_dict(results['ts2vec'], orient='index').rename_axis('sample').reset_index()
femba_df = pd.DataFrame.from_dict(results['femba'], orient='index').rename_axis('sample').reset_index()

model_training_report_df =  classic_mean_df.merge(
                                            ts2vec_df,
                                            left_on='sample',
                                            right_on='sample',
                                            suffixes=('_classic_mean', '_ts2vec')
                                        ).merge(
                                            femba_df,
                                            left_on='sample',
                                            right_on='sample',
                                            suffixes=('', '_femba'),
                                        ).rename(columns={
                                            'train_time_classic_mean': 'classic_mean_train_time',
                                            'train_time_ts2vec': 'ts2vec_train_time',
                                            'train_time': 'femba_train_time'
                                        }).round(3)

print(model_training_report_df.to_string(index=False, justify='center', float_format="{:.3f}".format))

 sample  classic_mean_train_time  ts2vec_train_time  femba_train_time
    1            6.224                 15.808             15.376     
    2            5.972                 15.206             15.013     
    3            6.659                 15.079             14.994     
    4            6.084                 15.348             15.350     
    5            5.944                 15.495             15.512     
    6            6.887                 15.454             15.619     
    7            6.435                 16.272             15.660     
    8            6.115                 15.594             15.476     
    9            6.519                 15.063             14.991     
   10            6.467                 15.463             15.360     


Exporting the final model training report

In [19]:
model_training_report_df.to_csv(
    model_training_report_path,
    index=False
)